# Day Trading — 데이터 수집 + LightGBM 모델 학습
Colab에서 실행. 셀 순서대로 실행하면 됩니다.
완료 후 Drive에 `model_largecap.txt` 등 4개 모델 파일이 저장됩니다.

In [ ]:
# [0] 설치
!pip install -q yfinance pandas-ta lightgbm pyarrow requests

In [ ]:
# [0-2] Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [1] CONFIG — 여기서 파라미터 조정
import os

LARGE_CAP_USD  = 17_000_000_000   # 25조 KRW 기준
MID_CAP_USD    =  1_400_000_000   # 2조 KRW 기준
INTERVAL       = '1h'
PERIOD         = '2y'
# 미국장은 하루 6.5시간. 6봉(6h)이면 오버나이트 갭이 레이블에 포함됨 → 3봉으로 제한
FORWARD_BARS   = 3                # 3h 후 수익률로 레이블 생성
# 스프레드 + 슬리피지 + 수수료 감안. 소형주일수록 비용이 크므로 여유 있게 설정
RETURN_THRESH  = 0.008            # 0.8% 이상이면 매수=1
LGBM_THRESHOLD = 0.6              # 예측 확률 임계값
TOP_FEATURES   = 30               # 중요도 상위 N개 피처만 사용
BATCH_MC       = 100              # 시총 수집 배치 크기
BATCH_PRICE    = 50               # 가격 수집 배치 크기
DELAY_SEC      = 1.0              # 배치 간 딜레이(초)

DRIVE_PATH     = '/content/drive/MyDrive/stock_trading/'
MC_CACHE       = DRIVE_PATH + 'market_caps.json'
PRICE_DIR      = DRIVE_PATH + 'price_1h/'
MODEL_DIR      = DRIVE_PATH + 'models/'

os.makedirs(PRICE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print('Drive 경로 준비 완료')

In [ ]:
# [2] 유니버스 수집 (NASDAQ Trader)
import requests
import pandas as pd
from io import StringIO
import re

URL = 'https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt'
r = requests.get(URL, headers={'User-Agent': 'stock-research-bot hideinthecodes@gmail.com'}, timeout=30)
df_uni = pd.read_csv(StringIO(r.text), sep='|')[:-1]  # 마지막 행 제거

# ETF·테스트 종목 제외
df_uni = df_uni[(df_uni['ETF'] == 'N') & (df_uni['Test Issue'] == 'N')]

# 특수문자 포함 티커 제외 (워런트, 유닛 등)
df_uni = df_uni[~df_uni['Symbol'].str.contains(r'[.\-+^$]', regex=True, na=True)]

# Rights / Units / Warrants / SPAC 등 제외
_excl = (
    '- Rights|- Units|- Warrant|Depositary Share|'
    'Acquisition Corp|Acquisition Inc|Blank Check|'
    '- Class A Ordinary|- Class B Ordinary'
)
df_uni = df_uni[~df_uni['Security Name'].str.contains(_excl, case=False, na=False, regex=True)]

ALL_TICKERS = df_uni['Symbol'].tolist()
print(f'유니버스: {len(ALL_TICKERS):,}개 종목')

In [ ]:
# [3] 시가총액 배치 수집 (체크포인트 + 병렬)
import json
import time
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed

# 캐시 로드
if os.path.exists(MC_CACHE):
    with open(MC_CACHE) as f:
        market_caps = json.load(f)
    print(f'캐시 로드: {len(market_caps):,}개')
else:
    market_caps = {}

todo = [t for t in ALL_TICKERS if t not in market_caps]
print(f'수집 필요: {len(todo):,}개')


def _fetch_mc(ticker: str) -> tuple[str, float]:
    try:
        mc = yf.Ticker(ticker).fast_info.get('market_cap', None)
        return ticker, float(mc) if mc else 0.0
    except Exception:
        return ticker, 0.0


for i in range(0, len(todo), BATCH_MC):
    batch = todo[i:i + BATCH_MC]
    # 병렬 수집 (10 workers → 순차 대비 약 5~8배 빠름)
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(_fetch_mc, t): t for t in batch}
        for fut in as_completed(futures):
            ticker, mc = fut.result()
            market_caps[ticker] = mc

    # 체크포인트 저장 (배치마다)
    with open(MC_CACHE, 'w') as f:
        json.dump(market_caps, f)
    pct = min(i + BATCH_MC, len(todo))
    print(f'  시총 수집 {pct}/{len(todo)} ({pct/len(todo)*100:.1f}%)')
    time.sleep(DELAY_SEC)

print(f'시총 수집 완료: {len(market_caps):,}개')

In [ ]:
# [3-2] 케이스별 티커 분류
large_tickers     = [t for t, mc in market_caps.items() if mc and mc >= LARGE_CAP_USD]
mid_tickers       = [t for t, mc in market_caps.items() if mc and MID_CAP_USD <= mc < LARGE_CAP_USD]
small_tickers     = [t for t, mc in market_caps.items() if mc and mc < MID_CAP_USD]
large_mid_tickers = large_tickers + mid_tickers
all_tickers_valid = [t for t, mc in market_caps.items() if mc and mc > 0]

CASES = {
    'largecap':   large_tickers,
    'large_mid':  large_mid_tickers,
    'smallcap':   small_tickers,
    'all':        all_tickers_valid,
}

for name, tickers in CASES.items():
    print(f'{name:12s}: {len(tickers):,}개')

In [ ]:
# [4] 1시간봉 OHLCV 수집 (체크포인트)
# 전체 유니버스 중 유효한 종목만 수집
import warnings
warnings.filterwarnings('ignore')

need_download = [
    t for t in all_tickers_valid
    if not os.path.exists(PRICE_DIR + f'{t}.parquet')
]
print(f'다운로드 필요: {len(need_download):,}개')

for i in range(0, len(need_download), BATCH_PRICE):
    batch = need_download[i:i + BATCH_PRICE]
    try:
        raw = yf.download(
            batch,
            period=PERIOD,
            interval=INTERVAL,
            auto_adjust=True,
            progress=False,
            threads=True,
        )
        if raw.empty:
            continue
        # 다중 티커면 MultiIndex, 단일이면 단순 컬럼
        if isinstance(raw.columns, pd.MultiIndex):
            for ticker in batch:
                try:
                    df_t = raw.xs(ticker, axis=1, level=1).dropna(how='all')
                    if len(df_t) > 100:
                        df_t.to_parquet(PRICE_DIR + f'{ticker}.parquet')
                except Exception:
                    pass
        else:
            ticker = batch[0]
            if len(raw) > 100:
                raw.to_parquet(PRICE_DIR + f'{ticker}.parquet')
    except Exception as e:
        print(f'  배치 {i} 실패: {e}')

    pct = min(i + BATCH_PRICE, len(need_download))
    if (i // BATCH_PRICE) % 10 == 0:
        print(f'  가격 수집 {pct}/{len(need_download)} ({pct/len(need_download)*100:.1f}%)')
    time.sleep(DELAY_SEC)

downloaded = [f.replace('.parquet','') for f in os.listdir(PRICE_DIR) if f.endswith('.parquet')]
print(f'수집 완료: {len(downloaded):,}개')

In [ ]:
# [4-2] 수집된 데이터 길이 분포 확인
# median이 100 근처면 데이터 부족 → INTERVAL을 '1d'로 변경 고려
import os
import pandas as pd

lengths = []
for f in os.listdir(PRICE_DIR):
    if not f.endswith('.parquet'):
        continue
    try:
        df_tmp = pd.read_parquet(PRICE_DIR + f)
        lengths.append(len(df_tmp))
    except Exception:
        pass

if lengths:
    s = pd.Series(lengths)
    print(f'수집된 파일: {len(lengths):,}개')
    print(s.describe().to_string())
    short = (s < 200).sum()
    print(f'\n200봉 미만 (학습 제외 대상): {short:,}개 ({short/len(s)*100:.1f}%)')
    if s.median() < 300:
        print('\n⚠️  중앙값이 300봉 미만입니다.')
        print('   yfinance 1h 데이터가 충분하지 않을 수 있습니다.')
        print("   CONFIG의 INTERVAL을 '1d'로 변경하면 더 많은 데이터를 수집할 수 있습니다.")
else:
    print('수집된 파일 없음')

In [ ]:
# [5] 피처 엔지니어링
import pandas_ta as ta
import numpy as np

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """pandas-ta 기반 피처 생성. OHLCV DataFrame 입력."""
    df = df.copy()
    df.columns = [c.lower() for c in df.columns]

    # 모멘텀
    df.ta.rsi(length=14, append=True)
    df.ta.macd(fast=12, slow=26, signal=9, append=True)
    df.ta.stoch(k=14, d=3, append=True)

    # 변동성
    df.ta.atr(length=14, append=True)
    df.ta.bbands(length=20, std=2, append=True)
    df.ta.adx(length=14, append=True)

    # 추세
    df.ta.ema(length=9,  append=True)
    df.ta.ema(length=21, append=True)
    df.ta.ema(length=50, append=True)
    # VWAP 제거: UTC strip 후에도 날짜 경계가 ET 기준이 아니어서 부정확

    # 거래량
    df.ta.obv(append=True)
    vol_ema = df['volume'].ewm(span=20).mean()
    df['vol_ratio'] = df['volume'] / (vol_ema + 1e-9)

    # 가격 패턴
    df['ret_1']     = df['close'].pct_change(1)
    df['ret_3']     = df['close'].pct_change(3)
    df['ret_6']     = df['close'].pct_change(6)
    df['gap']       = (df['open'] - df['close'].shift(1)) / (df['close'].shift(1) + 1e-9)
    df['hl_ratio']  = (df['high'] - df['low']) / (df['close'] + 1e-9)
    df['close_pos'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-9)

    # VWAP 대체: 종가 / EMA21 비율 (추세 대비 가격 위치)
    ema21_col = [c for c in df.columns if 'ema' in c.lower() and '21' in c]
    if ema21_col:
        df['price_to_ema21'] = df['close'] / (df[ema21_col[0]] + 1e-9)

    # ATR 정규화 (변동성 비율)
    atr_col = [c for c in df.columns if 'atr' in c.lower() and 'r_' not in c.lower()]
    if atr_col:
        df['atr_pct'] = df[atr_col[0]] / (df['close'] + 1e-9)

    return df


def make_label(df: pd.DataFrame, forward: int = FORWARD_BARS, thresh: float = RETURN_THRESH) -> pd.Series:
    """Forward N봉 수익률 기반 이진 레이블.
    FORWARD_BARS=3 → 3h 후. 미국장(6.5h) 기준 오버나이트 갭 최소화.
    thresh=0.8% — 스프레드/슬리피지/수수료 커버 가능한 수익만 양성으로 잡음.
    """
    fwd_ret = df['close'].shift(-forward) / df['close'] - 1
    return (fwd_ret > thresh).astype(int)


print('피처 함수 정의 완료')

In [ ]:
# [6] 케이스별 데이터셋 빌드 함수

def build_dataset(tickers: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    """주어진 티커 목록에서 피처+레이블 데이터셋 생성."""
    frames = []
    ok, fail = 0, 0
    for ticker in tickers:
        path = PRICE_DIR + f'{ticker}.parquet'
        if not os.path.exists(path):
            fail += 1
            continue
        try:
            df = pd.read_parquet(path)
            df = make_features(df)
            df['label'] = make_label(df)
            df = df.dropna()
            if len(df) < 200:
                fail += 1
                continue
            drop_cols = ['open','high','low','close','volume','label']
            feat_cols = [c for c in df.columns if c not in drop_cols]
            df['_ticker'] = ticker
            frames.append(df[feat_cols + ['label', '_ticker']])
            ok += 1
        except Exception:
            fail += 1

    print(f'  로드 성공: {ok}, 실패/스킵: {fail}')
    if not frames:
        raise ValueError('데이터 없음')

    # ignore_index=True 제거 → datetime index 유지
    # sort_index()가 시간 순 정렬을 해야 70/15/15 분할이 올바름
    combined = pd.concat(frames)
    combined = combined.sort_index()
    print(f'  index dtype: {combined.index.dtype}')  # datetime64[ns] 이어야 함

    feat_cols = [c for c in combined.columns if c not in ['label', '_ticker']]
    X = combined[feat_cols].astype(float)
    y = combined['label']
    return X, y


print('데이터셋 빌드 함수 정의 완료')

In [ ]:
# [7] LightGBM 학습 함수
import lightgbm as lgb
from sklearn.metrics import classification_report

def train_model(X: pd.DataFrame, y: pd.Series, case_name: str) -> lgb.Booster:
    # 날짜 기준 70/15/15 분할 (iloc 기준이면 같은 타임스탬프 데이터가 섞임)
    unique_dates = X.index.unique().sort_values()
    n = len(unique_dates)
    cut_train = unique_dates[int(n * 0.70)]
    cut_val   = unique_dates[int(n * 0.85)]

    X_train = X[X.index <  cut_train];  y_train = y[y.index <  cut_train]
    X_val   = X[(X.index >= cut_train) & (X.index < cut_val)]
    y_val   = y[(y.index >= cut_train) & (y.index < cut_val)]
    X_test  = X[X.index >= cut_val];   y_test  = y[y.index >= cut_val]

    pos_ratio = y_train.mean()
    print(f'  [{case_name}] train={len(X_train):,} val={len(X_val):,} test={len(X_test):,}')
    print(f'  분할 기준: ~{cut_train.date()} / ~{cut_val.date()} | pos_ratio={pos_ratio:.3f}')

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

    params = {
        'objective':        'binary',
        'metric':           'binary_logloss',
        'boosting_type':    'gbdt',
        'num_leaves':       63,
        'learning_rate':    0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq':     5,
        'is_unbalance':     True,
        'verbose':          -1,
        'n_jobs':           -1,
    }

    model = lgb.train(
        params, dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100),
        ],
    )

    # 피처 중요도 상위 TOP_FEATURES만 남기고 재학습
    fi = pd.Series(model.feature_importance('gain'), index=X_train.columns)
    top_feats = fi.nlargest(TOP_FEATURES).index.tolist()

    dtrain2 = lgb.Dataset(X_train[top_feats], label=y_train)
    dval2   = lgb.Dataset(X_val[top_feats],   label=y_val, reference=dtrain2)
    model2 = lgb.train(
        params, dtrain2,
        num_boost_round=1000,
        valid_sets=[dval2],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100),
        ],
    )

    # 테스트 평가
    proba = model2.predict(X_test[top_feats])
    pred  = (proba > LGBM_THRESHOLD).astype(int)
    print(classification_report(y_test, pred, digits=3))

    save_path = MODEL_DIR + f'model_{case_name}.txt'
    model2.save_model(save_path)

    feat_path = MODEL_DIR + f'features_{case_name}.json'
    with open(feat_path, 'w') as f:
        json.dump(top_feats, f)

    print(f'  저장 완료 → {save_path}')
    return model2


print('학습 함수 정의 완료')

In [ ]:
# [8] 케이스별 학습 실행
import psutil
import random

RUN_CASES = ['largecap', 'large_mid', 'smallcap', 'all']

trained_models = {}

for case_name in RUN_CASES:
    tickers = CASES[case_name]
    tickers = [t for t in tickers if os.path.exists(PRICE_DIR + f'{t}.parquet')]
    print(f'\n=== [{case_name}] {len(tickers):,}개 종목 ===')

    if len(tickers) == 0:
        print('  데이터 없음, 스킵')
        continue

    mem_gb = psutil.virtual_memory().available / 1e9
    print(f'  가용 메모리: {mem_gb:.1f} GB')
    MAX_TICKERS_PER_GB = 80
    safe_limit = int(mem_gb * MAX_TICKERS_PER_GB)
    if len(tickers) > safe_limit:
        print(f'  ⚠️  메모리 부족 위험 → {safe_limit}개로 샘플링')
        random.seed(42)
        tickers = random.sample(tickers, safe_limit)

    X, y = build_dataset(tickers)
    model = train_model(X, y, case_name)
    trained_models[case_name] = model
    del X, y

print('\n모든 케이스 학습 완료')

In [ ]:
# [9] 피처 중요도 시각화
import matplotlib.pyplot as plt

for case_name, model in trained_models.items():
    fi = pd.Series(
        model.feature_importance('gain'),
        index=model.feature_name(),
    ).nlargest(20)

    fig, ax = plt.subplots(figsize=(8, 5))
    fi[::-1].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Feature Importance — {case_name}')
    ax.set_xlabel('Gain')
    plt.tight_layout()
    plt.savefig(MODEL_DIR + f'feature_importance_{case_name}.png', dpi=100)
    plt.show()
    print(f'{case_name} 완료')

In [ ]:
# [10] 저장 결과 확인
print('Drive에 저장된 모델 파일:')
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(MODEL_DIR + f) / 1024
    print(f'  {f:40s}  {size:.1f} KB')

print()
print('로컬에서 로드하는 방법:')
print("  import lightgbm as lgb, json")
print("  model = lgb.Booster(model_file='model_largecap.txt')")
print("  feats = json.load(open('features_largecap.json'))")
print("  pred  = model.predict(X[feats])")